# Technical Validation — conservation checks on the delivered files

Reads each `data/final/`/`data/interim/.../nested_mriot_<year>.parquet` and checks the accounting identities (row/column balance, world-block conservation, state-share consistency) reported in the *Technical Validation* section.


In [ ]:
from paths import ROOT
import pandas as pd
import numpy as np

version_windc = "v3.1_RAS"


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG -- adapt if needed
# ══════════════════════════════════════════════════════════════════════════════
NESTED_ROOT = ROOT / f"data/interim/IOT/nested_mriot_{version_windc}"
OECD_AGG    = ROOT / "data/interim/IOT/OCDE ICIO aggregated"

EXTRA_ROWS = ["OUT", "TLS", "VA"]
FD_CATS    = {"DPABR", "GFCF", "GGFC", "HFCE", "INVNT", "NPISH"}
# World balance tolerance : published OECD have a native residual of ~100 M$ max
TOL_WORLD  = 200.0    # M$
TOL_REL    = 1e-6     # for the conservation checks (exact nesting)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def find_oecd_file(year):
    for folder in OECD_AGG.iterdir():
        for f in folder.glob(f"{year}_*.parquet"):
            return f
    raise FileNotFoundError(f"OECD {year} introuvable")

def _is_state(l):      return "_" in l and len(l.split("_")[0]) == 2
def _is_world(l):      return "_" in l and len(l.split("_")[0]) == 3
def _is_sector_col(c): return "_" in c and c.split("_",1)[1] not in FD_CATS
def _is_fd_col(c):     return "_" in c and c.split("_",1)[1] in FD_CATS

def _result(name, ok, detail=""):
    return ok, f"  {'✓' if ok else '✗'} {name}  →  {detail}"


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════
def check_nested_mriot(year, verbose=True):
    results = []

    nested_path = NESTED_ROOT / f"nested_mriot_{year}.parquet"
    if not nested_path.exists():
        print(f"[{year}] file missing")
        return None

    df  = pd.read_parquet(nested_path)
    dfo = pd.read_parquet(find_oecd_file(year))

    # ── Row / column partition ───────────────────────────────────────
    ind_rows       = [r for r in df.index   if r not in EXTRA_ROWS]
    world_rows     = [r for r in ind_rows   if _is_world(r)]
    state_rows     = [r for r in ind_rows   if _is_state(r)]
    world_sec_cols = [c for c in df.columns if _is_world(c) and _is_sector_col(c)]
    state_sec_cols = [c for c in df.columns if _is_state(c) and _is_sector_col(c)]
    world_fd_cols  = [c for c in df.columns if _is_world(c) and _is_fd_col(c)]
    state_fd_cols  = [c for c in df.columns if _is_state(c) and _is_fd_col(c)]
    all_sec_cols   = world_sec_cols + state_sec_cols
    all_fd_cols    = world_fd_cols  + state_fd_cols

    states  = sorted({r.split("_")[0] for r in state_rows})
    sectors = [r.split("_",1)[1] for r in df.index if r.startswith(states[0]+"_")]
    n_s, n_sec, n_w = len(states), len(sectors), len(world_rows)
    n_ws = len(world_sec_cols)

    # Blocs numpy
    Z       = df.loc[ind_rows,  all_sec_cols].values.astype(float)
    F       = df.loc[ind_rows,  all_fd_cols ].values.astype(float)
    OUT_col = df.loc[ind_rows,  "OUT"       ].values.astype(float)
    OUT_row = df.loc["OUT",     all_sec_cols].values.astype(float)
    TLS     = df.loc["TLS",     all_sec_cols].values.astype(float)
    VA      = df.loc["VA",      all_sec_cols].values.astype(float)

    # OECD reference (without USA)
    usa_oecd_rows         = [r for r in dfo.index   if r.startswith("USA_")]
    oecd_world_rows_ref   = [r for r in dfo.index   if _is_world(r) and not r.startswith("USA_")]
    oecd_non_usa_sec_cols = [c for c in dfo.columns if _is_world(c) and _is_sector_col(c) and not c.startswith("USA_")]
    oecd_non_usa_fd_cols  = [c for c in dfo.columns if _is_world(c) and _is_fd_col(c)     and not c.startswith("USA_")]
    usa_sec_cols_oecd     = [f"USA_{s}" for s in sectors if f"USA_{s}" in dfo.columns]

    if verbose:
        print(f"\n{'═'*65}")
        print(f"  NESTED MRIO CHECK -- {year}")
        print(f"  {n_w} world rows + {n_s}×{n_sec} state rows = {len(ind_rows)} total")
        print(f"{'═'*65}")

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 1 -- Row balance (world) :  Σ_j Z_ij + Σ_f F_if = OUT_col_i
    # For the states, the imbalance is expected (WinDC != OECD)
    # ══════════════════════════════════════════════════════════════════════
    use_row = Z.sum(axis=1) + F.sum(axis=1)
    err1    = use_row - OUT_col

    err1_w  = err1[:n_w]
    err1_s  = err1[n_w:]
    ok1 = np.abs(err1_w).max() < TOL_WORLD
    results.append(_result(
        "Équilibre ligne — monde   (Σ uses = OUT_col)",
        ok1,
        f"max|ε|={np.abs(err1_w).max():.2f} M$  "
        f"rel={( np.abs(err1_w)/(np.abs(OUT_col[:n_w])+1e-12)).max():.2e}"
    ))
    ok1_s = np.abs(err1_s).max() < TOL_WORLD
    results.append(_result(
        "Équilibre ligne — états (post-harmonisation)",
        ok1_s,
        f"max|ε|={np.abs(err1_s).max():.2f} M$  "
        f"rel={(np.abs(err1_s)/(np.abs(OUT_col[n_w:])+1e-12)).max():.2e}"
    ))
    # ══════════════════════════════════════════════════════════════════════
    # CHECK 2 -- Column balance (world) :  Σ_i Z_ij + TLS_j + VA_j = OUT_row_j
    # ══════════════════════════════════════════════════════════════════════
    supply_col = Z.sum(axis=0) + TLS + VA
    err2       = supply_col - OUT_row

    err2_w = err2[:n_ws]
    err2_s = err2[n_ws:]
    ok2 = np.abs(err2_w).max() < TOL_WORLD
    results.append(_result(
        "Équilibre colonne — monde (Σ inputs + VA + TLS = OUT_row)",
        ok2,
        f"max|ε|={np.abs(err2_w).max():.2f} M$  "
        f"rel={( np.abs(err2_w)/(np.abs(OUT_row[:n_ws])+1e-12)).max():.2e}"
    ))
    ok2_s = np.abs(err2_s).max() < TOL_WORLD
    results.append(_result(
        "Équilibre colonne — états (post-harmonisation)",
        ok2_s,
        f"max|ε|={np.abs(err2_s).max():.2f} M$  "
        f"rel={( np.abs(err2_s)/(np.abs(OUT_row[n_ws:])+1e-12)).max():.2e}"
    ))
    # ══════════════════════════════════════════════════════════════════════
    # CHECK 3 -- OUT symmetry  (OUT_col_i = OUT_row_i)
    # ══════════════════════════════════════════════════════════════════════
    err3 = OUT_col - OUT_row
    ok3  = np.abs(err3).max() < 1e-6
    results.append(_result(
        "Symétrie OUT_col = OUT_row",
        ok3,
        f"max|ε|={np.abs(err3).max():.2e}"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 4a -- Conservation States->World
    # Σ_states Z_nested[state_i, world_j] = Z_oecd[USA_i, world_j]
    # ══════════════════════════════════════════════════════════════════════
    Z_oecd_uw   = dfo.loc[usa_oecd_rows,       oecd_non_usa_sec_cols].values.astype(float)
    Z_sw_nested = df.loc[state_rows,            world_sec_cols       ].values.astype(float)
    Z_sw_sum    = Z_sw_nested.reshape(n_s, n_sec, -1).sum(axis=0)
    err4a       = Z_sw_sum - Z_oecd_uw
    rel4a       = (np.abs(err4a) / (np.abs(Z_oecd_uw) + 1e-12)).max()
    ok4a = rel4a < TOL_REL
    results.append(_result(
        "Conservation États→Monde  (Σ_s Z[s_i, w_j] = Z_oecd[USA_i, w_j])",
        ok4a,
        f"max|ε|={np.abs(err4a).max():.2e} M$  rel={rel4a:.2e}"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 4b -- Conservation World->States
    # Σ_states Z_nested[world_i, state_j] = Z_oecd[world_i, USA_j]
    # ══════════════════════════════════════════════════════════════════════
    Z_oecd_wu   = dfo.loc[oecd_world_rows_ref, usa_sec_cols_oecd].values.astype(float)
    Z_ws_nested = df.loc[world_rows,           state_sec_cols   ].values.astype(float)
    Z_ws_sum    = Z_ws_nested.reshape(-1, n_s, n_sec).sum(axis=1)
    err4b       = Z_ws_sum - Z_oecd_wu
    rel4b       = (np.abs(err4b) / (np.abs(Z_oecd_wu) + 1e-12)).max()
    ok4b        = rel4b < TOL_REL * 1000
    results.append(_result(
        "Conservation Monde→États  (Σ_s Z[w_i, s_j] = Z_oecd[w_i, USA_j])",
        ok4b,
        f"max|ε|={np.abs(err4b).max():.2e} M$  rel={rel4b:.2e}"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 4c -- Conservation FD States->World
    # ══════════════════════════════════════════════════════════════════════
    F_oecd_uw   = dfo.loc[usa_oecd_rows, oecd_non_usa_fd_cols].values.astype(float)
    F_sw_nested = df.loc[state_rows,     world_fd_cols        ].values.astype(float)
    F_sw_sum    = F_sw_nested.reshape(n_s, n_sec, -1).sum(axis=0)
    err4c       = F_sw_sum - F_oecd_uw
    rel4c       = (np.abs(err4c) / (np.abs(F_oecd_uw) + 1e-12)).max()
    ok4c        = rel4c < TOL_REL * 1000
    results.append(_result(
        "Conservation FD États→Monde  (Σ_s F[s_i, fd_w] = F_oecd[USA_i, fd_w])",
        ok4c,
        f"max|ε|={np.abs(err4c).max():.2e} M$  rel={rel4c:.2e}"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 5 -- Non-negativity of Z
    # ══════════════════════════════════════════════════════════════════════
    n_neg = (Z < -1e-3).sum()
    ok5   = n_neg == 0
    results.append(_result(
        "Non-négativité de Z",
        ok5,
        f"n_négatifs={n_neg}  min={Z.min():.4g} M$"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 6 -- World excluding USA unchanged (Z_ww bit-by-bit)
    # ══════════════════════════════════════════════════════════════════════
    Z_ww_ref = dfo.loc[oecd_world_rows_ref, oecd_non_usa_sec_cols].values.astype(float)
    Z_ww_new = df.loc[world_rows,           world_sec_cols        ].values.astype(float)
    max_diff6 = np.abs(Z_ww_new - Z_ww_ref).max()
    ok6 = max_diff6 < 1e-6
    results.append(_result(
        "Monde hors USA inchangé  (Z_ww identique à OECD)",
        ok6,
        f"max|diff|={max_diff6:.2e}"
    ))

    # ══════════════════════════════════════════════════════════════════════
    # CHECK 7 — Divergence Z_ss (WinDC) vs Z_oecd[USA,USA]  [info]
    # ══════════════════════════════════════════════════════════════════════
    Z_oecd_uu   = dfo.loc[usa_oecd_rows, usa_sec_cols_oecd].values.astype(float)
    Z_ss_nested = df.loc[state_rows,     state_sec_cols   ].values.astype(float)
    Z_ss_agg    = Z_ss_nested.reshape(n_s, n_sec, n_s, n_sec).sum(axis=(0, 2))
    rel7 = np.linalg.norm(Z_ss_agg - Z_oecd_uu) / (np.linalg.norm(Z_oecd_uu) + 1e-12)
    results.append(_result(
        "Z_ss WinDC vs OECD[USA,USA]  [info]",
        True,
        f"||diff||/||ref||={rel7:.3e}  (écart attendu : sources BEA ≠ OCDE)"
    ))
    # ══════════════════════════════════════════════════════════════════════
    # CHECK 8 — Total output mondial vs OECD  [info]
    # ══════════════════════════════════════════════════════════════════════
    oecd_all_ind = [r for r in dfo.index if r not in EXTRA_ROWS]
    tot_oecd   = dfo.loc[oecd_all_ind, "OUT"].values.astype(float).sum()
    tot_nested = OUT_col.sum()
    rel8 = abs(tot_nested - tot_oecd) / (abs(tot_oecd) + 1e-12)
    results.append(_result(
        "Total output mondial vs OECD  [info]",
        True,
        f"OECD={tot_oecd:.0f}  nested={tot_nested:.0f}  "
        f"Δ={abs(tot_nested-tot_oecd):.0f} M$ ({100*rel8:.2f}%)"
    ))

    # -- Summary ----------------------------------------------------------
    hard_checks = [(ok, l) for ok, l in results if "info" not in l and "Déséquilibre" not in l]
    n_ok   = sum(ok for ok, _ in hard_checks)
    n_fail = sum(not ok for ok, _ in hard_checks)

    if verbose:
        print()
        for _, line in results:
            print(line)
        print(f"\n  {'─'*63}")
        print(f"  Checks stricts : {n_ok}/{len(hard_checks)} OK"
              + (f"  ← {n_fail} ÉCHEC(S)" if n_fail else ""))

    return {
        "year"                    : year,
        "checks_stricts_ok"       : n_ok,
        "checks_stricts_total"    : len(hard_checks),
        "equilibre_ligne_monde_max_abs"  : np.abs(err1_w).max(),
        "equilibre_col_monde_max_abs"    : np.abs(err2_w).max(),
        "conservation_sw_max_rel" : rel4a,
        "conservation_ws_max_rel" : rel4b,
        "conservation_fd_max_rel" : rel4c,
        "z_ss_divergence_norm"    : rel7,
        "output_mondial_rel_diff" : rel8,
        "desequilibre_etats_max"  : np.abs(err1_s).max(),
    }

    

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VERIFICATION OVER ALL AVAILABLE YEARS
# ══════════════════════════════════════════════════════════════════════════════
summary = []
for f in sorted(NESTED_ROOT.glob("nested_mriot_*.parquet")):
    year = int(f.stem.split("_")[-1])
    row = check_nested_mriot(year, verbose=True)
    if row:
        summary.append(row)

if len(summary) > 1:
    print("\n\n== MULTI-YEAR SUMMARY ==")
    print(pd.DataFrame(summary).set_index("year").round(6).to_string())
